In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [27]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Stacked_CCSS")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
Epilepsy_Combined.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- 80601-8: string (nullable = true)
 |-- 80602-6: string (nullable = true)
 |-- 8061-4: string (nullable = true)
 |-- 8065-5: string (nullable = true)
 |-- 80658-8: string (nullable = true)
 |-- 80660-4: string (nullable = true)
 |-- 8067-1: string (nullable = true)
 |-- 80676-0: string (nullable = true)
 |-- 80679-4: string (nullable = true)
 |-- 8068-9: string (nullable = true)
 |-- 80680-2: string (nullable = true)
 |-- 80681-0: string (nullable = true)
 |-- 80682-8: string (nullable = true)
 |-- 80685-1: string (nullable = true)
 |-- 807-8: string (nullable = true)
 |-- 8071-3: string (nullable = true)
 |-- 8072-1: string (nullable = true)
 |-- 8074-7: string (nullable = true)
 |-- 8092-9: string (nullable = true)
 |-- 8094-5: string (nullable = true)
 |-- 8095-2: string (nullable = true)
 |-- 8097-8: string (nullable = true)
 |-- 8098-6: string (nullable = true)
 |-- 8099-4: string (nullable = true)
 |-- 8101-8: string (nullable = tr

In [28]:
from pyspark.sql.functions import col

# Current column order
current_columns = Epilepsy_Combined.columns
print("Current Columns:", current_columns)

# Find the index position of the "personid" column
personid_index = current_columns.index("personid")

# Define the range of columns to move
start_column = "birthdate"
end_column = "mh_date"
start_index = current_columns.index(start_column)
end_index = current_columns.index(end_column)

# Rearrange columns
rearranged_columns = (
    current_columns[:personid_index + 1] +  # Columns before "personid"
    current_columns[start_index:end_index + 1] +  # Columns between "birthdate" and "mh_date"
    current_columns[personid_index + 1:start_index] +  # Columns after "personid" but before "birthdate"
    current_columns[end_index + 1:]  # Columns after "mh_date"
)

# Select and rearrange columns in the DataFrame
rearranged_df = Epilepsy_Combined.select(*rearranged_columns)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '80685-1', '807-8', '8071-3', '8072-1', '8074-7', '8092-9', '8094-5', '8095-2', '8097-8', '8098-6', '8099-4', '8101-8', '8103-4', '8107-5', '8110-9', '8112-5', '8115-8', '81155-4', '8116-6', '8117-4', '8118-2', '8119-0', '81201-6', '8122-4', '8123-2', '8124-0', '8126-5', '8127-3', '8128-1', '81285-9', '813-6', '8130-7', '81308-9', '81309-7', '8131-5', '8132-3', '8133-1', '8135-6', '814-4', '8150-5', '81622-3', '81623-1', '81641-3', '81655-3', '8169-5', '817-7', '8170-3', '8172-9', '81788-2', '81790-8', '8191-9', '820-1', '8214-9', '82159-5', '82160-3', '82161-1', '82162-9', '82163-7', '82164-5', '82165-2', '82166-0', '82167-8', '82168-6', '82169-4', '8217-2', '82170-2', '82171-0', '82172-8', '82173-6', '82174-4', '82175-1', '82176-9', '82177-7', '82178-5', '82179-3', '82181-9', '82182-7', '82183-5', '82184-3', '82185-0'

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '80685-1', '807-8', '8071-3', '8072-1', '8074-7', '8092-9', '8094-5', '8095-2', '8097-8', '8098-6', '8099-4', '8101-8', '8103-4', '8107-5', '8110-9', '8112-5', '8115-8', '81155-4', '8116-6', '8117-4', '8118-2', '8119-0', '81201-6', '8122-4', '8123-2', '8124-0', '8126-5', '8127-3', '8128-1', '81285-9', '813-6', '8130-7', '81308-9', '81309-7', '8131-5', '8132-3', '8133-1', '8135-6', '814-4', '8150-5', '81622-3', '81623-1', '81641-3', '81655-3', '8169-5', '817-7', '8170-3', '8172-9', '81788-2', '81790-8', '8191-9', '820-1', '8214-9', '82159-5', '82160-3', '82161-1', '82162-9', '82163-7', '82164-5', '82165-2', '82166-0', '82167-8', '82168-6', '82169-4', '8217-2', '82170-2', '82171-0', '82172-8', '82173-6', '82174-4'

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------+-------+------+------+-------+-------+------+-------+-------+------+-------+-------+-------+-------+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+-------+------+------+------+------+------+------+-------+-----+------+-------+-------+------+------+------+------+-----+------+-------+-------+-------+-------+------+-----+------+------+-------+-------+------+-----+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

<IPython.core.display.Javascript object>

In [12]:
# Print column names and their corresponding indices
for index, col_name in enumerate(Epilepsy_Combined.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column 80601-8 has index 1
Column 80602-6 has index 2
Column 8061-4 has index 3
Column 8065-5 has index 4
Column 80658-8 has index 5
Column 80660-4 has index 6
Column 8067-1 has index 7
Column 80676-0 has index 8
Column 80679-4 has index 9
Column 8068-9 has index 10
Column 80680-2 has index 11
Column 80681-0 has index 12
Column 80682-8 has index 13
Column 80685-1 has index 14
Column 807-8 has index 15
Column 8071-3 has index 16
Column 8072-1 has index 17
Column 8074-7 has index 18
Column 8092-9 has index 19
Column 8094-5 has index 20
Column 8095-2 has index 21
Column 8097-8 has index 22
Column 8098-6 has index 23
Column 8099-4 has index 24
Column 8101-8 has index 25
Column 8103-4 has index 26
Column 8107-5 has index 27
Column 8110-9 has index 28
Column 8112-5 has index 29
Column 8115-8 has index 30
Column 81155-4 has index 31
Column 8116-6 has index 32
Column 8117-4 has index 33
Column 8118-2 has index 34
Column 8119-0 has index 35
Column 81201-6 has index 3

In [13]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column 80601-8 has index 9
Column 80602-6 has index 10
Column 8061-4 has index 11
Column 8065-5 has index 12
Column 80658-8 has index 13
Column 80660-4 has index 14
Column 8067-1 has index 15
Column 80676-0 has index 16
Column 80679-4 has index 17
Column 8068-9 has index 18
Column 80680-2 has index 19
Column 80681-0 has index 20
Column 80682-8 has index 21
Column 80685-1 has index 22
Column 807-8 has index 23
Column 8071-3 has index 24
Column 8072-1 has index 25
Column 8074-7 has index 26
Column 8092-9 has index 27
Column 8094-5 has index 28
Column 8095-2 has index 29
Column 8097-8 has index 30
Column 8098-6 has index 31
Column 8099-4 has index 32
Column 8101-8 has index 33
Column 8103-4 has index 34
Column 8107-5 has in

In [29]:
# Current column order
current_columns = rearranged_df.columns
print("Current Columns:", current_columns)

# Define the index positions of mh_date and latest_diagdate
mh_date_index = 8
latest_diagdate_index = 259

# Rearrange columns
rearranged_df = rearranged_df.select(
    *current_columns[:mh_date_index + 1],  # Columns before mh_date
    current_columns[latest_diagdate_index],  # latest_diagdate
    *current_columns[mh_date_index + 1:latest_diagdate_index],  # Columns between mh_date and latest_diagdate
    *current_columns[latest_diagdate_index + 1:]  # Columns after latest_diagdate
)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '80685-1', '807-8', '8071-3', '8072-1', '8074-7', '8092-9', '8094-5', '8095-2', '8097-8', '8098-6', '8099-4', '8101-8', '8103-4', '8107-5', '8110-9', '8112-5', '8115-8', '81155-4', '8116-6', '8117-4', '8118-2', '8119-0', '81201-6', '8122-4', '8123-2', '8124-0', '8126-5', '8127-3', '8128-1', '81285-9', '813-6', '8130-7', '81308-9', '81309-7', '8131-5', '8132-3', '8133-1', '8135-6', '814-4', '8150-5', '81622-3', '81623-1', '81641-3', '81655-3', '8169-5', '817-7', '8170-3', '8172-9', '81788-2', '81790-8', '8191-9', '820-1', '8214-9', '82159-5', '82160-3', '82161-1', '82162-9', '82163-7', '82164-5', '82165-2', '82166-0', '82167-8', '82168-6', '82169-4', '8217-2', '82170-2', '82171-0', '82172-8', '82173-6', '8217

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '80685-1', '807-8', '8071-3', '8072-1', '8074-7', '8092-9', '8094-5', '8095-2', '8097-8', '8098-6', '8099-4', '8101-8', '8103-4', '8107-5', '8110-9', '8112-5', '8115-8', '81155-4', '8116-6', '8117-4', '8118-2', '8119-0', '81201-6', '8122-4', '8123-2', '8124-0', '8126-5', '8127-3', '8128-1', '81285-9', '813-6', '8130-7', '81308-9', '81309-7', '8131-5', '8132-3', '8133-1', '8135-6', '814-4', '8150-5', '81622-3', '81623-1', '81641-3', '81655-3', '8169-5', '817-7', '8170-3', '8172-9', '81788-2', '81790-8', '8191-9', '820-1', '8214-9', '82159-5', '82160-3', '82161-1', '82162-9', '82163-7', '82164-5', '82165-2', '82166-0', '82167-8', '82168-6', '82169-4', '8217-2', '82170-2', '82171-0', '82172-8', '

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------------------------+-------+-------+------+------+-------+-------+------+-------+-------+------+-------+-------+-------+-------+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+-------+------+------+------+------+------+------+-------+-----+------+-------+-------+------+------+------+------+-----+------+-------+-------+-------+-------+------+-----+------+------+-------+-------+------+-----+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---

<IPython.core.display.Javascript object>

In [15]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column latest_diagdate has index 9
Column 80601-8 has index 10
Column 80602-6 has index 11
Column 8061-4 has index 12
Column 8065-5 has index 13
Column 80658-8 has index 14
Column 80660-4 has index 15
Column 8067-1 has index 16
Column 80676-0 has index 17
Column 80679-4 has index 18
Column 8068-9 has index 19
Column 80680-2 has index 20
Column 80681-0 has index 21
Column 80682-8 has index 22
Column 80685-1 has index 23
Column 807-8 has index 24
Column 8071-3 has index 25
Column 8072-1 has index 26
Column 8074-7 has index 27
Column 8092-9 has index 28
Column 8094-5 has index 29
Column 8095-2 has index 30
Column 8097-8 has index 31
Column 8098-6 has index 32
Column 8099-4 has index 33
Column 8101-8 has index 34
Column 8103

In [30]:
# Current column order
current_columns = rearranged_df.columns
print("Current Columns:", current_columns)

# Define the index positions
latest_diagdate_index = 9
start_index = 109
end_index = 208

# Rearrange columns
rearranged_df = rearranged_df.select(
    *current_columns[:latest_diagdate_index + 1],  # Columns before latest_diagdate
    *current_columns[start_index:end_index + 1],  # Columns to move
    *current_columns[latest_diagdate_index + 1:start_index],  # Columns after latest_diagdate and before moved columns
    *current_columns[end_index + 1:]  # Columns after moved columns
)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '80685-1', '807-8', '8071-3', '8072-1', '8074-7', '8092-9', '8094-5', '8095-2', '8097-8', '8098-6', '8099-4', '8101-8', '8103-4', '8107-5', '8110-9', '8112-5', '8115-8', '81155-4', '8116-6', '8117-4', '8118-2', '8119-0', '81201-6', '8122-4', '8123-2', '8124-0', '8126-5', '8127-3', '8128-1', '81285-9', '813-6', '8130-7', '81308-9', '81309-7', '8131-5', '8132-3', '8133-1', '8135-6', '814-4', '8150-5', '81622-3', '81623-1', '81641-3', '81655-3', '8169-5', '817-7', '8170-3', '8172-9', '81788-2', '81790-8', '8191-9', '820-1', '8214-9', '82159-5', '82160-3', '82161-1', '82162-9', '82163-7', '82164-5', '82165-2', '82166-0', '82167-8', '82168-6', '82169-4', '8217-2', '82170-2', '82171-0', '82172-8

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8', '8

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-------+-------+------+------+-------+-------+------+-------+-------+------+-------+-------+-------+-------+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+-------+------+------+------+------+------+------+-------+-----+------+-------+-------+------+------+------+------+-----+------+-------+-------+-------+-------+

<IPython.core.display.Javascript object>

In [22]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column latest_diagdate has index 9
Column Z21 has index 10
Column M19 has index 11
Column S68 has index 12
Column Y30 has index 13
Column B05 has index 14
Column A23 has index 15
Column H82 has index 16
Column V89 has index 17
Column I31 has index 18
Column V72 has index 19
Column R16 has index 20
Column Q61 has index 21
Column O12 has index 22
Column X76 has index 23
Column Z12 has index 24
Column S39 has index 25
Column L65 has index 26
Column F25 has index 27
Column G12 has index 28
Column E02 has index 29
Column X04 has index 30
Column B79 has index 31
Column F32 has index 32
Column M54 has index 33
Column B34 has index 34
Column S60 has index 35
Column Z19 has index 36
Column T36 has index 37
Column E44 has index 38

In [31]:
# Current column order
current_columns = rearranged_df.columns
print("Current Columns:", current_columns)

# Define the index positions of B99 and the range (260 to 359)
b99_index = 109
start_index = 260
end_index = 359

# Rearrange columns
rearranged_df = rearranged_df.select(
    *current_columns[:b99_index + 1],  # Columns before B99
    *current_columns[start_index:end_index + 1],  # Columns within the range
    *current_columns[b99_index + 1:start_index],  # Columns between B99 and the range
    *current_columns[end_index + 1:]  # Columns after the range
)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', '80601-8', '80602-6', '8061-4', '8065-5', '80658-8', '80660-4', '8067-1', '80676-0', '80679-4', '8068-9', '80680-2', '80681-0', '80682-8'

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'CP4', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A', '

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-------+-------

<IPython.core.display.Javascript object>

In [24]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column latest_diagdate has index 9
Column Z21 has index 10
Column M19 has index 11
Column S68 has index 12
Column Y30 has index 13
Column B05 has index 14
Column A23 has index 15
Column H82 has index 16
Column V89 has index 17
Column I31 has index 18
Column V72 has index 19
Column R16 has index 20
Column Q61 has index 21
Column O12 has index 22
Column X76 has index 23
Column Z12 has index 24
Column S39 has index 25
Column L65 has index 26
Column F25 has index 27
Column G12 has index 28
Column E02 has index 29
Column X04 has index 30
Column B79 has index 31
Column F32 has index 32
Column M54 has index 33
Column B34 has index 34
Column S60 has index 35
Column Z19 has index 36
Column T36 has index 37
Column E44 has index 38

In [32]:
# Current column order
current_columns = rearranged_df.columns
print("Current Columns:", current_columns)

# Define the index positions of K58 and the range to move
k58_index = 209
start_move_index = 309
end_move_index = 561

# Rearrange columns
rearranged_df = rearranged_df.select(
    *current_columns[:k58_index + 1],  # Columns before K58
    *current_columns[start_move_index:end_move_index + 1],  # Columns to move
    *current_columns[k58_index + 1:start_move_index],  # Columns between K58 and the range to move
    *current_columns[end_move_index + 1:]  # Columns after the range to move
)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'CP4', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'CP4', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A', '

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---------------

<IPython.core.display.Javascript object>

In [35]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column latest_diagdate has index 9
Column Z21 has index 10
Column M19 has index 11
Column S68 has index 12
Column Y30 has index 13
Column B05 has index 14
Column A23 has index 15
Column H82 has index 16
Column V89 has index 17
Column I31 has index 18
Column V72 has index 19
Column R16 has index 20
Column Q61 has index 21
Column O12 has index 22
Column X76 has index 23
Column Z12 has index 24
Column S39 has index 25
Column L65 has index 26
Column F25 has index 27
Column G12 has index 28
Column E02 has index 29
Column X04 has index 30
Column B79 has index 31
Column F32 has index 32
Column M54 has index 33
Column B34 has index 34
Column S60 has index 35
Column Z19 has index 36
Column T36 has index 37
Column E44 has index 38

In [37]:
# Check if 'label' column exists in rearranged_df
if 'label' in rearranged_df.columns:
    label_index = rearranged_df.columns.index('label')
    print("rearranged_df contains the 'label' column at index position:", label_index)
else:
    print("rearranged_df does not contain the 'label' column.")


▸,:,


rearranged_df contains the 'label' column at index position: 260


In [38]:
# Assuming rearranged_df is your DataFrame

# Current column order
current_columns = rearranged_df.columns
print("Current Columns:", current_columns)

# Define the index positions
label_index = 260
target_column_index = 811

# Rearrange columns
rearranged_df = rearranged_df.select(
    *current_columns[:label_index],  # Columns before the label
    *current_columns[label_index + 1:target_column_index + 1],  # Columns between label and target column
    current_columns[label_index],  # label column
    *current_columns[target_column_index + 1:]  # Columns after the target column
)

# New column order
new_columns = rearranged_df.columns
print("New Columns:", new_columns)

# Show DataFrame
rearranged_df.show(truncate=False)

▸,:,


Current Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'CP4', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A

New Columns: ['personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'mh_date', 'latest_diagdate', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'CP4', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A', '

<IPython.core.display.Javascript object>

+------------------------------------+----------+--------+--------------------------------+--------------------+--------------------+----------+------+-------+-------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
# Print column names and their corresponding indices
for index, col_name in enumerate(rearranged_df.columns):
    print(f"Column {col_name} has index {index}")

▸,:,


Column personid has index 0
Column birthdate has index 1
Column EPI_date has index 2
Column TBI_date has index 3
Column age_of_TBI_diagnosis has index 4
Column age_at_EPI_diagnosis has index 5
Column race has index 6
Column gender has index 7
Column mh_date has index 8
Column latest_diagdate has index 9
Column Z21 has index 10
Column M19 has index 11
Column S68 has index 12
Column Y30 has index 13
Column B05 has index 14
Column A23 has index 15
Column H82 has index 16
Column V89 has index 17
Column I31 has index 18
Column V72 has index 19
Column R16 has index 20
Column Q61 has index 21
Column O12 has index 22
Column X76 has index 23
Column Z12 has index 24
Column S39 has index 25
Column L65 has index 26
Column F25 has index 27
Column G12 has index 28
Column E02 has index 29
Column X04 has index 30
Column B79 has index 31
Column F32 has index 32
Column M54 has index 33
Column B34 has index 34
Column S60 has index 35
Column Z19 has index 36
Column T36 has index 37
Column E44 has index 38

In [40]:
print(rearranged_df.count())

▸,:,


<IPython.core.display.Javascript object>

12000


<IPython.core.display.Javascript object>

In [ ]:
rearranged_df.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_SmallSet')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>